# 06 · Harness RFT: coding episodes in the local-ai coding agent (generation)

Round 2 data. The RFT-Agent model solves coding tasks **inside the coding agent of
[local-ai](https://github.com/salihyesil59/local-ai)**, the app it runs in at home: the same system prompt, the same
13 tools (`plan`, `edit_file`, `verify`, `search`, ...), the same plan → verify → review → judge workflow.
The model learns this workflow from its own successful runs.

**How it runs:**
- llama.cpp serves the **same Q4 GGUF that runs at home** (from Hugging Face) as an OpenAI-compatible server, with
  many parallel slots. No merge, no 16-bit model: the data comes from exactly the model it will improve.
- `local-ai traces collect` runs the coding agent on many KodCode tasks at once against that server. Each run is
  graded by the task's tests, with the test file restored first (editing the tests does not help).
- Every run leaves a trace with the model's thinking. The good runs become `harness_sft.jsonl`, which `04_sft`
  trains on together with the round-1 and agent-1 data (04 trains a new LoRA from the base model, as in round 1).

**Selection (same rules as 03, plus one):**
- Only runs that finished, passed the restored tests and fit the training limit of **10240 tokens**; shortest first.
- **Clean episodes only:** no message from the program (an automatic verify, a nudge) after the model's first reply.
  The chat template drops the thinking of every reply before the last user message, so a reply before such a message
  would be trained without its thinking.

**Runtime and cost:**
- Runtime: **A100 80GB** (≈6.8 CU/hour): the ~22 GB model and the context of all slots on the GPU.
- llama.cpp build ~10 min, download ~5 min, generation at most `MAX_GEN_HOURS`. **~25 CU** in total.
- Resumable: traces and results are on Drive; after a disconnect run the notebook again, it goes on where it stopped.

**Before you start:**
- `colab-ai/eval/` on Drive (for `agent.py` and the held-out `tasks/agent_code.jsonl`, which stays out of the pool).

In [ ]:
!pip install -q pytest datasets
!pip install -q "local-ai @ git+https://github.com/salihyesil59/local-ai"
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# llama.cpp server, CUDA build for this GPU only (~10 min)
!git clone --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp
!cmake -S /content/llama.cpp -B /content/llama.cpp/build -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=native -DLLAMA_CURL=OFF -DCMAKE_BUILD_TYPE=Release > /dev/null
!cmake --build /content/llama.cpp/build -j --target llama-server 2>&1 | tail -3

## Settings

In [ ]:
DRIVE_ROOT = "/content/drive/MyDrive/colab-ai"
GGUF_REPO = "salihyesil59/Qwen3.6-35B-A3B-RFT-Agent-GGUF"   # the model that runs at home
TOKENIZER = "Qwen/Qwen3.6-35B-A3B-FP8"                       # chat template and tokenizer for the selection (as in 04)
SERVED_NAME = "rft-agent"
SLOTS = 32                  # parallel conversations on the server
SLOT_CONTEXT = 20480        # tokens per conversation (prompt + reply)
CU_PER_HOUR = 6.77
SEED = 3407

EVAL_DIR = f"{DRIVE_ROOT}/eval"
ROUND1_GEN = f"{DRIVE_ROOT}/rft/round1/generations.jsonl"
AGENT1_EPISODES = f"{DRIVE_ROOT}/rft/agent1/episodes.jsonl"
OUT_DIR = f"{DRIVE_ROOT}/rft/harness1"

# Generation
POOL = 400                  # KodCode tasks (medium/hard), disjoint from the eval set, round 1 and agent 1
SAMPLES = 2                 # runs per task
CONCURRENCY = SLOTS         # runs at the same time (each is a multi-turn conversation)
TEMPERATURE = 0.8
MAX_TURNS = 30
CONTEXT_CHARS = 24000       # the agent compacts above this: every segment stays short enough for training
MINUTES = 20                # time limit of one run
MAX_GEN_HOURS = 3.0

# Selection
TRAIN_MAX = 10240
EASY_KEEP, MAX_PER_TASK = 0.5, 2

In [ ]:
import os, sys, json, glob, random, time, hashlib, importlib, collections, subprocess
from google.colab import drive, userdata
from huggingface_hub import login, hf_hub_download, list_repo_files

drive.mount("/content/drive")
login(token=userdata.get("HF_TOKEN"))
os.makedirs(OUT_DIR, exist_ok=True)
assert os.path.exists(f"{EVAL_DIR}/agent.py"), f"{EVAL_DIR}/agent.py not found: copy the repo's current eval/ folder to Drive"
sys.path.insert(0, EVAL_DIR)
sys.path_importer_cache.pop(EVAL_DIR, None)
importlib.invalidate_caches()
import agent as A


def uid(prefix, text):
    return f"{prefix}-{hashlib.md5(text.encode()).hexdigest()[:12]}"

## Model (llama.cpp server)

In [ ]:
import requests

# The quantized model of the repo (not the imatrix file, not the vision projector)
name = next(f for f in list_repo_files(GGUF_REPO) if f.endswith(".gguf") and "Q4" in f and "mmproj" not in f)
MODEL_PATH = hf_hub_download(GGUF_REPO, name, local_dir="/content/gguf")
print("model:", MODEL_PATH)

log = open("/content/llama-server.log", "w")
server = subprocess.Popen(
    ["/content/llama.cpp/build/bin/llama-server", "-m", MODEL_PATH, "-a", SERVED_NAME,
     "--host", "127.0.0.1", "--port", "8000",
     "-ngl", "999",                                  # the whole model on the GPU (at home the experts are on the CPU)
     "-np", str(SLOTS), "-c", str(SLOTS * SLOT_CONTEXT),   # the context is shared out among the slots
     "-fa", "on",
     "--jinja",                                      # the model's own chat template: tool calls as XML
     "--reasoning-format", "deepseek",               # the thinking comes back apart from the answer
     "--seed", str(SEED)],
    stdout=log, stderr=subprocess.STDOUT)
for _ in range(120):
    if server.poll() is not None:
        raise RuntimeError("llama-server stopped:\n" + open("/content/llama-server.log").read()[-3000:])
    try:
        if requests.get("http://127.0.0.1:8000/health", timeout=5).ok:
            break
    except requests.RequestException:
        pass
    time.sleep(10)
print(requests.get("http://127.0.0.1:8000/v1/models").json()["data"][0]["id"], "is ready")

In [ ]:
# Check: a tool call and the thinking come back as separate fields (the coding agent needs both)
from openai import OpenAI

client = OpenAI(base_url="http://127.0.0.1:8000/v1", api_key="local")
tool = {"type": "function", "function": {"name": "add", "description": "Add two integers.", "parameters": {
    "type": "object", "properties": {"a": {"type": "integer"}, "b": {"type": "integer"}}, "required": ["a", "b"]}}}
t0 = time.time()
r = client.chat.completions.create(model=SERVED_NAME, messages=[{"role": "user", "content": "Use the tool: 17 + 25"}],
                                   tools=[tool], temperature=0.6, max_tokens=2048)
m = r.choices[0].message
print(f"{r.usage.completion_tokens / (time.time() - t0):.0f} tok/s for one request")
print("tool calls:", m.tool_calls)
print("thinking:", (getattr(m, "reasoning_content", None) or getattr(m, "reasoning", None) or "")[:300])
assert m.tool_calls, "no tool call parsed: see /content/llama-server.log (the coding agent can still read calls from text)"

## KodCode tasks

- The same filter as the coding pool of 03 (instruct style, medium/hard, functions imported from `solution`).
- Not in the held-out eval set, not in round 1, not in agent 1: new problems for this round.
- A task is a folder with `solution.py` (the signatures) and `test_solution.py`; the tests are restored before grading.

In [ ]:
from datasets import load_dataset

TASKS = f"{OUT_DIR}/tasks.jsonl"
PROMPT = ("{question}\n\nThe project folder has solution.py with the signatures of the functions to write, and "
          "test_solution.py with tests for them. Implement the functions in solution.py; do not change the tests.")
held_out = lambda qid: int(hashlib.md5(qid.encode()).hexdigest()[:2], 16) < 8   # the eval split of 03
if os.path.exists(TASKS):
    tasks = [json.loads(l) for l in open(TASKS, encoding="utf-8")]
else:
    seen = set()
    if os.path.exists(ROUND1_GEN):
        seen |= {json.loads(l)["id"] for l in open(ROUND1_GEN, encoding="utf-8")}
    if os.path.exists(AGENT1_EPISODES):
        seen |= {json.loads(l)["id"] for l in open(AGENT1_EPISODES, encoding="utf-8")}
    tasks = []
    stream = load_dataset("KodCode/KodCode-V1", split="train", streaming=True).shuffle(seed=SEED + 2, buffer_size=20_000)
    for r in stream:
        if (r["style"] != "instruct" or r["gpt_difficulty"] not in ("medium", "hard")
                or not r["test_info"] or "from solution import" not in r["test"] or held_out(r["question_id"])):
            continue
        stub = A.make_stub(r["test_info"])
        qid = r["question_id"]
        if not stub or uid("code", qid) in seen or uid("agent", qid) in seen:
            continue
        tasks.append({"id": uid("harness", qid), "prompt": PROMPT.format(question=r["question"]),
                      "files": {"solution.py": stub, "test_solution.py": r["test"]},
                      "protected": ["test_solution.py"],
                      "grade": "python -m pytest -q -p no:cacheprovider test_solution.py"})
        if len(tasks) >= POOL:
            break
    with open(TASKS, "w", encoding="utf-8") as f:
        f.writelines(json.dumps(t, ensure_ascii=False) + "\n" for t in tasks)
print(f"tasks: {len(tasks)}")
print(tasks[0]["prompt"][:600])

## Generation (resumable)

Smoke test first: two tasks, one run each, into a scratch folder. Then the whole pool. Runs already graded are
skipped, so after a disconnect just run the cells again.

In [ ]:
with open("/content/smoke.jsonl", "w", encoding="utf-8") as f:
    f.writelines(json.dumps(t, ensure_ascii=False) + "\n" for t in tasks[:2])
!rm -rf /content/smoke
!local-ai traces collect /content/smoke.jsonl --out /content/smoke --base-url http://127.0.0.1:8000/v1 --model {SERVED_NAME} --samples 1 --concurrency 2 --max-turns {MAX_TURNS} --context-chars {CONTEXT_CHARS}
run = next(json.loads(l) for p in glob.glob("/content/smoke/traces/*.jsonl") for l in open(p) if '"segments"' in l)
replies = [m for m in run["segments"][-1] if m["role"] == "assistant"]
print(f"{len(replies)} replies, {sum(bool(m.get('reasoning')) for m in replies)} with thinking; ended: {run['reason']}")

In [ ]:
t0 = time.time()
!local-ai traces collect {TASKS} --out {OUT_DIR}/collect --work /content/work --base-url http://127.0.0.1:8000/v1 --model {SERVED_NAME} --samples {SAMPLES} --concurrency {CONCURRENCY} --temperature {TEMPERATURE} --max-turns {MAX_TURNS} --context-chars {CONTEXT_CHARS} --minutes {MINUTES} --hours {MAX_GEN_HOURS}
print(f"Time: {(time.time() - t0) / 60:.1f} min ≈ {(time.time() - t0) / 3600 * CU_PER_HOUR:.1f} CU")

## Selection → SFT data

- `local-ai traces export` takes the runs that finished, were not rejected by the judge and passed the restored tests;
  every segment of a run (cut by a compaction) is a sample.
- Here: clean episodes only (see the top), rendered with the model's own chat template, at most `TRAIN_MAX` tokens;
  the shortest per task, up to `MAX_PER_TASK` from partially solved tasks, one from `EASY_KEEP` of the fully solved.
- Output: `harness_sft.jsonl` in the format of `agent_sft.jsonl` (tool call arguments as objects).

In [ ]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(TOKENIZER)
!local-ai traces export --from {OUT_DIR}/collect/traces --out {OUT_DIR}/samples.jsonl
samples = [json.loads(l) for l in open(f"{OUT_DIR}/samples.jsonl", encoding="utf-8")]
results = [json.loads(l) for l in open(f"{OUT_DIR}/collect/results.jsonl", encoding="utf-8")]
runs_per_task = collections.Counter(r["id"] for r in results)
solved_per_task = collections.Counter(r["id"] for r in results if r["resolved"])


def task_id(sample):
    return sample["meta"]["source"].removesuffix(".trace.jsonl").rsplit("-", 1)[0]   # "<id>-<k>.trace.jsonl"


def clean(messages):
    first = next(i for i, m in enumerate(messages) if m["role"] == "assistant")
    return all(m["role"] != "user" for m in messages[first:])


def for_template(messages):
    out = []
    for m in messages:
        m = {k: v for k, v in m.items() if not (k == "reasoning_content" and not v)}
        if m.get("tool_calls"):
            m["tool_calls"] = [{**c, "function": {**c["function"], "arguments": json.loads(c["function"]["arguments"] or "{}")}}
                               for c in m["tool_calls"]]
        out.append(m)
    return out


rng = random.Random(SEED)
stats = collections.Counter()
by_task = collections.defaultdict(list)
for s in samples:
    stats["samples"] += 1
    if not clean(s["messages"]):
        continue
    stats["clean"] += 1
    messages = for_template(s["messages"])
    n = len(tok(tok.apply_chat_template(messages, tools=s["tools"], tokenize=False), add_special_tokens=False).input_ids)
    if n > TRAIN_MAX:
        continue
    stats["fitting"] += 1
    by_task[task_id(s)].append((n, {"messages": messages, "tools": s["tools"], "source": "harness", "n_tokens": n}))
examples = []
for tid, good in by_task.items():
    good.sort(key=lambda x: x[0])
    if solved_per_task[tid] == runs_per_task[tid]:
        if rng.random() > EASY_KEEP:
            continue
        good = good[:1]
    else:
        good = good[:MAX_PER_TASK]
    examples += [ex for _, ex in good]
rng.shuffle(examples)
with open(f"{OUT_DIR}/harness_sft.jsonl", "w", encoding="utf-8") as f:
    f.writelines(json.dumps(ex, ensure_ascii=False) + "\n" for ex in examples)

print(f"tasks run: {len(runs_per_task)}, solved at least once: {len(solved_per_task)}, "
      f"runs resolved: {sum(solved_per_task.values())}/{len(results)}")
print(f"samples: {stats['samples']}, clean: {stats['clean']}, fitting {TRAIN_MAX}: {stats['fitting']}, kept: {len(examples)}")
if examples:
    print(f"avg tokens: {sum(e['n_tokens'] for e in examples) / len(examples):.0f} -> {OUT_DIR}/harness_sft.jsonl")

In [ ]:
# Check: a rendered example (thinking + tool calls in the agent's own tools)
if examples:
    print(tok.apply_chat_template(examples[0]["messages"], tools=examples[0]["tools"], tokenize=False)[-3000:])

## Finish: stop spending CU

In [ ]:
server.terminate()
from google.colab import runtime
runtime.unassign()